# Fulfillment operations: analyst walkthrough
This notebook uses **synthetic data only**. Run `python generate_data.py` from the repository root before opening.
We assess staffing coverage, productivity, service, and backlog across three simulated facilities.

In [ ]:
import csv
from pathlib import Path
from collections import defaultdict
import matplotlib.pyplot as plt
data_path = Path('data/fulfillment_operations.csv')
if not data_path.exists():
    data_path = Path('../data/fulfillment_operations.csv')
with data_path.open(newline='') as f:
    rows = list(csv.DictReader(f))
len(rows), sorted({r['facility'] for r in rows})

## Weighted facility KPIs
Use totals in the numerator and denominator instead of averaging daily rates.

In [ ]:
summary = {}
for facility in sorted({r['facility'] for r in rows}):
    group = [r for r in rows if r['facility'] == facility]
    total = lambda key: sum(int(r[key]) for r in group)
    summary[facility] = {
        'on_time_pct': 100 * total('on_time_orders') / total('orders_processed'),
        'orders_per_labor_hour': total('orders_processed') / total('actual_labor_hours'),
        'staffing_coverage_pct': 100 * total('actual_staff') / total('scheduled_staff'),
        'ending_backlog': int(group[-1]['backlog_orders']),
    }
summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
facilities = list(summary)
axes[0].bar(facilities, [summary[f]['on_time_pct'] for f in facilities])
axes[0].set(title='On-time fulfillment', ylabel='Percent', ylim=(0, 100))
axes[1].bar(facilities, [summary[f]['ending_backlog'] for f in facilities])
axes[1].set(title='Ending backlog', ylabel='Orders')
plt.tight_layout()
plt.show()

## Interpretation and decisions
See [BUSINESS_INSIGHTS.md](../BUSINESS_INSIGHTS.md) for a concise executive summary and three recommendations. These are **descriptive associations within a programmed simulation**, not causal or real-world business findings.